In [ ]:
import sys
from pathlib import Path

import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from ripplegw.waveforms.cbc.Taylor.TaylorF2 import gen_TaylorF2

sys.path.append("../src")
from datos import cargar
from filtro_adaptado import parecido, psd_en
from plantillas import generador, taylorf2

In [ ]:
EVENTO_PSD = "GW150914"
DATOS = Path("../datos")
DURACION = 128
DURACION_FFT = 4
DURACION_SEGMENTO = 256
F_MIN = 20
F_MAX = 1024
F_REFERENCIA = 20
COMPARACIONES = [(1.2, 0.25), (1.5, 0.2), (6.0, 0.19), (8.5, 0.22), (30.0, 0.25)]
ORDENES = [0, 2, 3, 4, 5, 6, 7]
CASOS_ORDENES = [(1.2, 0.25), (8.5, 0.22)]
MASAS_FUSION = [1.2, 2, 3, 4.5, 6, 8.5, 12, 18, 25, 30]
PARECIDO_OBJETIVO = 0.97

In [ ]:
datos_h1 = cargar("H1", EVENTO_PSD, DURACION, DATOS)
fs = int(datos_h1.sample_rate.value)
n = DURACION_SEGMENTO * fs
df = fs / n
indices = np.arange(int(F_MIN / df), int(F_MAX / df) + 1)
frecuencias = indices * df
n_parecido = 2 * indices[-1]
psd = psd_en(np.arange(n // 2 + 1) * df, datos_h1.value, fs, DURACION_FFT)

In [ ]:
for masa_chirp, eta in COMPARACIONES:
    mia = taylorf2(frecuencias, masa_chirp, eta)
    suya = np.asarray(gen_TaylorF2(jnp.array(frecuencias), jnp.array([masa_chirp, eta, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0]), F_REFERENCIA))
    hasta_isco = mia != 0
    cociente = mia[hasta_isco] / suya[hasta_isco]
    desfase = np.unwrap(np.angle(cociente))
    print(f"Mc {masa_chirp}, η {eta}: hasta {frecuencias[hasta_isco][-1]:.0f} Hz, amplitud mía / ripple − 1 hasta {np.abs(np.abs(cociente) - 1).max():.0e}, el desfase varía {desfase.max() - desfase.min():.0e} rad")

In [ ]:
for masa_chirp, eta in CASOS_ORDENES:
    completa = taylorf2(frecuencias, masa_chirp, eta)
    truncadas = np.array([taylorf2(frecuencias, masa_chirp, eta, orden) for orden in ORDENES])
    parecidos = parecido(truncadas, completa, psd, df, n_parecido, indices)
    print(f"Mc {masa_chirp}, η {eta}, parecido con la de 3.5PN: " + ", ".join(f"{orden / 2:g}PN {p:.3f}" for orden, p in zip(ORDENES, parecidos)))

In [ ]:
imr = generador(frecuencias, F_MIN)
parecidos_fusion = []
for masa_chirp in MASAS_FUSION:
    parecidos_fusion.append(parecido(taylorf2(frecuencias, masa_chirp, 0.25)[None, :], imr(masa_chirp, 0.25, 0.0), psd, df, n_parecido, indices)[0])
    print(f"Mc {masa_chirp}: TaylorF2 frente a IMRPhenomD, parecido {parecidos_fusion[-1]:.3f}")

In [ ]:
plt.semilogx(MASAS_FUSION, parecidos_fusion, "o-")
plt.axhline(PARECIDO_OBJETIVO, color="k", linestyle="--")
plt.xlabel(r"Masa chirp (M$_\odot$), q = 1, sin giro")
plt.ylabel("Parecido TaylorF2 – IMRPhenomD");